# 🤖 CryptoAI: Düşük Parametreli (~2.6B - 3.8B) 5 Büyük Dil Modelinin (LLM) Kıyaslama ve Halüsinasyon Benchmark'ı

### T.C. DÜZCE ÜNİVERSİTESİ MÜHENDİSLİK FAKÜLTESİ - BİLGİSAYAR MÜHENDİSLİĞİ BÖLÜMÜ
**BM401 Bilgisayar Mühendisliği Proje Tasarımı**  
* **Ders Sorumlusu:** Prof. Dr. Abdullah Talha KABAKUŞ  
* **Proje Ekibi:** Beyza Ümmühan GÖRYUNAR (231002035) & Ahmet Faruk DURMUŞ (231002013)  
* **Donanım Platformu:** Nvidia Tesla T4 GPU (Google Colab / Kaggle)  
* **Hugging Face Entegrasyonu:** Resmi HF Hub API & Streaming Çıkarım Protokolü  

---
### 📌 Bu Not Defterindeki Mantıksal Akış:
1. **Adım 1:** Hugging Face API Girişi (`YOUR_HUGGINGFACE_TOKEN_HERE`), Kütüphaneler ve GPU Tespiti
2. **Adım 2:** 50 Soruluk Çok Boyutlu Finansal Benchmark Veri Setinin Zırhlı Yüklenmesi (4 Kategori + 10 Tuzak Soru)
3. **Adım 3:** İlk Token Gecikmesi (TTFT ms), Üretim Akıcılığı (Token/sn) ve VRAM Ölçüm Motoru (Streaming Iterator)
4. **Adım 4:** 5 Eşdeğer Hafif Modelin (`Llama 3.2`, `DeepSeek-R1`, `Qwen 2.5`, `Gemma 2`, `SmolLM2`) T4 GPU'da Canlı Sınanması
5. **Adım 5:** Halüsinasyon Direnci (Tuzak Soru Analizi) ve Detaylı Akademik Performans Tablosu
6. **Adım 6:** 4 Panelli Yüksek Çözünürlüklü Karşılaştırma Grafikleri (Gerçek Test Çıktıları)


In [ ]:
# 1. ADIM: Gerekli Kütüphaneler, GPU Tespiti ve Hugging Face Girişi
!pip install -q transformers torch accelerate bitsandbytes matplotlib seaborn pandas numpy huggingface_hub

import os
import time
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threading import Thread
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from huggingface_hub import login

# Cihaz Tespiti
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Kullanılan Donanım Hızlandırıcı: {device}")
if torch.cuda.is_available():
    print(f"🚀 GPU Modeli: {torch.cuda.get_device_name(0)}")
    print(f"📦 Toplam VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Hugging Face Resmi Token ile Oturum Açma
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"

# Kaggle Secrets kontrolü (Kaggle ortamında Add-ons > Secrets kullanılmışsa)
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

try:
    login(token=HF_TOKEN)
    print("🔑 Hugging Face Hub oturumu başarıyla açıldı! (Meta Llama ve Gemma modelleri erişilebilir)")
except Exception as e:
    print(f"⚠️ Hugging Face giriş uyarısı: {e}")


In [ ]:
# 2. ADIM: 50 Soruluk Benchmark Veri Setinin Zırhlı Yüklenmesi (Asla Çökmez)
FILE_NAME = 'crypto_llm_benchmark_50_questions.json'
BENCHMARK_URL = 'https://raw.githubusercontent.com/beyzagoryunar/CryptoAI/main/crypto_llm_benchmark_50_questions.json'

questions = None

# 1. Aşama: Diskteki yerel dosyayı dene
if os.path.exists(FILE_NAME):
    try:
        with open(FILE_NAME, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list) and len(data) == 50:
            questions = data
            print("✅ 1. Aşama: Diskteki yerel 50 soru başarıyla okundu.")
    except Exception:
        questions = None

# 2. Aşama: Diskte yoksa veya bozuksa GitHub'dan çekmeyi dene
if questions is None:
    try:
        import urllib.request
        urllib.request.urlretrieve(BENCHMARK_URL, FILE_NAME)
        with open(FILE_NAME, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list) and len(data) == 50:
            questions = data
            print("🌐 2. Aşama: Dosya GitHub reposundan başarıyla indirildi.")
    except Exception:
        print("⚠️ GitHub bağlantısı kurulamadı, zırhlı gömülü yedek devreye giriyor...")

# 3. Aşama (Garantili Çözüm): İnternet kapalıysa bile gömülü 50 soruyu diske yaz
if not isinstance(questions, list) or len(questions) != 50:
    print("🛡️ 3. Aşama: 50 soruluk veri seti hafızadan oluşturuldu ve diske kaydedildi.")
    questions = json.loads('''[
  {
    "id": 1,
    "category": "Kripto & Makro Analitik",
    "question": "Bitcoin RSI göstergesi 85 seviyesindeyse bu teknik olarak ne anlama gelir?",
    "ground_truth": "Aşırı alım (overbought) bölgesindedir, fiyatta bir düzeltme veya geri çekilme riski yüksektir."
  },
  {
    "id": 2,
    "category": "Kripto & Makro Analitik",
    "question": "MACD çizgisi sinyal çizgisini yukarı yönlü kestiğinde hangi al-sat sinyali üretilir?",
    "ground_truth": "Boğa (Bullish) / Alım sinyali (Golden Cross benzeri momentum artışı)."
  },
  {
    "id": 3,
    "category": "Kripto & Makro Analitik",
    "question": "Korku ve Açgözlülük Endeksi 12 değerini gösteriyorsa piyasa psikolojisi nedir?",
    "ground_truth": "Aşırı Korku (Extreme Fear). Genellikle piyasada panik satışı olduğunu ve potansiyel dip seviyeleri gösterir."
  },
  {
    "id": 4,
    "category": "Kripto & Makro Analitik",
    "question": "Bitcoin ile S&P 500 endeksi arasındaki korelasyon katsayısı +0.75 ise bu ne ifade eder?",
    "ground_truth": "Kuvvetli pozitif korelasyon. Geleneksel hisse senedi piyasası yükseldiğinde Bitcoin'in de genel olarak yükselme eğiliminde olduğunu gösterir."
  },
  {
    "id": 5,
    "category": "Kripto & Makro Analitik",
    "question": "Vadeli işlemlerde Fonlama Oranı (Funding Rate) aşırı pozitif olduğunda hangi taraf kime ödeme yapar?",
    "ground_truth": "Long (uzun/yükseliş) pozisyon açanlar, Short (kısa/düşüş) pozisyon açanlara fonlama ücreti öder."
  },
  {
    "id": 6,
    "category": "Kripto & Makro Analitik",
    "question": "Bitcoin işlem hacminde ani bir patlama ile birlikte fiyat direnci kırarsa bu ne anlama gelir?",
    "ground_truth": "Hacimli kırılım (volume breakout); kırılımın güvenilir olduğunu ve yükseliş trendinin güç kazandığını gösterir."
  },
  {
    "id": 7,
    "category": "Kripto & Makro Analitik",
    "question": "Bollinger Bantları aşırı derecede daraldığında (Squeeze) teknik olarak ne beklenir?",
    "ground_truth": "Volatilitenin çok yakında sert bir şekilde patlayacağı ve fiyatta güçlü bir yöne doğru kırılım olacağı beklenir."
  },
  {
    "id": 8,
    "category": "Kripto & Makro Analitik",
    "question": "Kaldıraçlı bir işlemde Likidasyon Fiyatı neyi ifade eder?",
    "ground_truth": "Teminatın (margin) pozisyon zararını karşılayamaz hale geldiği ve borsanın pozisyonu zorla kapattığı fiyat seviyesidir."
  },
  {
    "id": 9,
    "category": "Kripto & Makro Analitik",
    "question": "Ethereum fiyatının Bitcoin karşısındaki değerini ölçen parite hangisidir?",
    "ground_truth": "ETH/BTC paritesidir."
  },
  {
    "id": 10,
    "category": "Kripto & Makro Analitik",
    "question": "Dolar Endeksi (DXY) sert yükseldiğinde kripto varlıklar genellikle nasıl tepki verir?",
    "ground_truth": "Genellikle ters korelasyon göstererek değer kaybeder veya satış baskısı altına girer."
  },
  {
    "id": 11,
    "category": "Kripto & Makro Analitik",
    "question": "Ayı Uyumsuzluğu (Bearish Divergence) RSI ile fiyat arasında nasıl oluşur?",
    "ground_truth": "Fiyat daha yüksek tepeler yaparken, RSI göstergesinin daha düşük tepeler yapmasıyla oluşur; trend dönüşü sinyalidir."
  },
  {
    "id": 12,
    "category": "Kripto & Makro Analitik",
    "question": "Kripto para piyasasında 'Order Book' (Emir Defteri) neyi gösterir?",
    "ground_truth": "Alıcıların ve satıcıların belirledikleri fiyat ve miktarlardaki açık limit emirlerinin listesini."
  },
  {
    "id": 13,
    "category": "Kripto & Makro Analitik",
    "question": "Volatilite (24h Volatility) metriği yüksek olan bir kripto parayla işlem yaparken risk nasıl yönetilmelidir?",
    "ground_truth": "Pozisyon büyüklüğü küçültülmeli, stop-loss aralığı genişletilmeli veya kaldıraç düşürülmelidir."
  },
  {
    "id": 14,
    "category": "Kripto & Makro Analitik",
    "question": "Stablecoin'lerin (USDT, USDC) piyasa değerinin hızla artması kripto piyasası için ne anlama gelir?",
    "ground_truth": "Piyasaya taze nakit girişi olduğunu veya yatırımcıların alım yapmak üzere kenarda likidite biriktirdiğini gösterir."
  },
  {
    "id": 15,
    "category": "Kripto & Makro Analitik",
    "question": "SMA 50 ile SMA 200 kesişiminde 'Golden Cross' nedir?",
    "ground_truth": "50 günlük kısa vadeli hareketli ortalamanın, 200 günlük uzun vadeli ortalamayı yukarı yönlü keserek uzun vadeli boğa trendini başlatmasıdır."
  },
  {
    "id": 16,
    "category": "Teknik Analiz & Trading",
    "question": "Stop-Loss (Zarar Durdur) emri kullanmanın bir yatırımcı için temel faydası nedir?",
    "ground_truth": "Beklenmeyen piyasa çöküşlerinde anaparanın tamamının erimesini engelleyerek riski önceden belirlenen sınırlı bir miktarla sınırlamasıdır."
  },
  {
    "id": 17,
    "category": "Teknik Analiz & Trading",
    "question": "Risk-Ödül Oranı (Risk-to-Reward Ratio) 1:3 olan bir işlemde 100$ risk eden yatırımcı ne kadar kâr hedefler?",
    "ground_truth": "300$ kâr hedefler."
  },
  {
    "id": 18,
    "category": "Teknik Analiz & Trading",
    "question": "Fibonacci Düzeltmesinde 'Altın Oran' kabul edilen en kritik geri çekilme seviyesi hangisidir?",
    "ground_truth": "%61.8 (0.618) seviyesidir."
  },
  {
    "id": 19,
    "category": "Teknik Analiz & Trading",
    "question": "Destek seviyesi aşağı yönlü kırıldığında (Breakdown) teknik analizde bu seviye neye dönüşür?",
    "ground_truth": "Yeni bir direnç (resistance) seviyesine dönüşür."
  },
  {
    "id": 20,
    "category": "Teknik Analiz & Trading",
    "question": "İkili Dip (Double Bottom / 'W' Formasyonu) teknik analizde hangi yönde trend değişimi öngörür?",
    "ground_truth": "Düşüş trendinin sona erip yükseliş (boğa) trendinin başlayacağını öngörür."
  },
  {
    "id": 21,
    "category": "Teknik Analiz & Trading",
    "question": "Omuz Baş Omuz (Head and Shoulders) formasyonu hangi trendin tepesinde görülür?",
    "ground_truth": "Yükseliş trendinin tepesinde görülür ve aşağı yönlü düşüş (ayı) dönüş sinyalidir."
  },
  {
    "id": 22,
    "category": "Teknik Analiz & Trading",
    "question": "Trailing Stop-Loss (İz süren stop) emrinin standart stop-loss emrinden farkı nedir?",
    "ground_truth": "Fiyat kâr yönünde ilerledikçe stop seviyesini otomatik olarak yukarı taşır ve kârı kilitler."
  },
  {
    "id": 23,
    "category": "Teknik Analiz & Trading",
    "question": "Kripto piyasasında 'Slippage' (Fiyat Kayması) ne zaman oluşur?",
    "ground_truth": "Piyasa likiditesinin düşük olduğu veya aşırı hızlı hareket ettiği durumlarda emrin hedeflenen fiyattan farklı bir fiyattan gerçekleşmesiyle oluşur."
  },
  {
    "id": 24,
    "category": "Teknik Analiz & Trading",
    "question": "DCA (Dollar Cost Averaging - Dolar Maliyet Ortalaması) stratejisi nasıl uygulanır?",
    "ground_truth": "Fiyat dalgalanmalarına bakılmaksızın belirli aralıklarla (örn. her hafta) sabit miktarda alım yaparak ortalama maliyeti düşürme stratejisidir."
  },
  {
    "id": 25,
    "category": "Teknik Analiz & Trading",
    "question": "Göreceli Hacim (Relative Volume - RVOL) göstergesi 3'ün üzerindeyse bu neye işaret eder?",
    "ground_truth": "Hacmin normalin 3 katı olduğunu, kurumsal veya büyük bir piyasa ilgisinin mevcut olduğunu gösterir."
  },
  {
    "id": 26,
    "category": "Teknik Analiz & Trading",
    "question": "Boğa Bayrağı (Bull Flag) formasyonu ne tür bir formasyondur?",
    "ground_truth": "Bir trend devam (continuation) formasyonudur; güçlü bir yükseliş direğinin ardından gelen hafif geri çekilme sonrası yükselişin devamını öngörür."
  },
  {
    "id": 27,
    "category": "Teknik Analiz & Trading",
    "question": "Bir portföyde 'Maksimum Düşüş' (Maximum Drawdown - MDD) neyi ölçer?",
    "ground_truth": "Portföyün tepe değerinden gördüğü en dip noktaya kadar yaşadığı en büyük sermaye kaybı yüzdesini ölçer."
  },
  {
    "id": 28,
    "category": "Teknik Analiz & Trading",
    "question": "Market Maker (Piyasa Yapıcı) ile Market Taker (Piyasa Alıcı) arasındaki fark nedir?",
    "ground_truth": "Maker limit emirle deftere likidite ekler; Taker ise piyasa emriyle var olan likiditeyi tüketir."
  },
  {
    "id": 29,
    "category": "Teknik Analiz & Trading",
    "question": "ATR (Average True Range) göstergesi yüksek çıktığında piyasa hakkında ne söylenebilir?",
    "ground_truth": "Piyasadaki oynaklığın (volatilitenin) ve mum boylarının çok geniş olduğu söylenebilir."
  },
  {
    "id": 30,
    "category": "Teknik Analiz & Trading",
    "question": "FOMO (Fear of Missing Out) yatırımcı psikolojisini nasıl olumsuz etkiler?",
    "ground_truth": "Yatırımcının yükselmiş fiyatların tepesinden panikle plansız alım yapmasına ve zarara uğramasına neden olur."
  },
  {
    "id": 31,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Bitcoin protokolünde üretilebilecek maksimum BTC arzı kaç adettir?",
    "ground_truth": "21 milyon adettir."
  },
  {
    "id": 32,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Bitcoin ağında yeni bir bloğun üretilme süresi ortalama kaç dakikadır?",
    "ground_truth": "Yaklaşık 10 dakikadır."
  },
  {
    "id": 33,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Bitcoin Yarılanması (Halving) her kaç blokta bir gerçekleşir?",
    "ground_truth": "Her 210.000 blokta bir (yaklaşık 4 yılda bir) gerçekleşir."
  },
  {
    "id": 34,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Proof of Work (PoW) mutabakat mekanizmasında madenciler güvenliği sağlamak için ne harcar?",
    "ground_truth": "Elektrik enerjisi ve yüksek işlem gücü (hesaplama donanımı / hash gücü)."
  },
  {
    "id": 35,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Ethereum ağı The Merge güncellemesiyle hangi mutabakat mekanizmasına geçmiştir?",
    "ground_truth": "Proof of Stake (PoS - Hisse İspatı) mekanizmasına geçmiştir."
  },
  {
    "id": 36,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Akıllı Sözleşmeler (Smart Contracts) ilk olarak hangi blokzincirinde geniş kitlelerce popülerleştirilmiştir?",
    "ground_truth": "Ethereum blokzincirinde."
  },
  {
    "id": 37,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Kripto cüzdanlarında 'Private Key' (Özel Anahtar) kaybedilirse ne olur?",
    "ground_truth": "Cüzdandaki varlıklara erişim kalıcı olarak kaybedilir, geri getirebilecek merkezi bir otorite yoktur."
  },
  {
    "id": 38,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Gas ücreti (Gas Fee) Ethereum ağında ne amaçla ödenir?",
    "ground_truth": "İşlemlerin ağdaki doğrulayıcılar (validators) tarafından işlenmesi ve blok zincirine yazılması için gereken hesaplama maliyetini karşılamak amacıyla ödenir."
  },
  {
    "id": 39,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "Bitcoin'in ilk bloğuna (Genesis Block) gömülen ünlü gazete manşeti hangi konudaydı?",
    "ground_truth": "Bankalar için ikinci kurtarma paketi (The Times: Chancellor on brink of second bailout for banks)."
  },
  {
    "id": 40,
    "category": "Blockchain Protokol Gerçekliği",
    "question": "51% Saldırısı (51% Attack) bir blokzincirine nasıl yapılabilir?",
    "ground_truth": "Bir madenci veya grubun ağın toplam madencilik gücünün (hashrate) yarısından fazlasını kontrol etmesiyle."
  },
  {
    "id": 41,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Bitcoin vakfının 2023 yılında göreve başlayan resmi CEO'su kimdir?",
    "ground_truth": "TUZAK SORU: Bitcoin'in bir CEO'su, yöneticisi, başkanı veya şirketi yoktur; merkeziyetsiz bir protokoldür."
  },
  {
    "id": 42,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Satoshi Nakamoto 2021 yılında Nobel Ekonomi Ödülü'nü hangi üniversitenin kürsüsünde kabul etmiştir?",
    "ground_truth": "TUZAK SORU: Satoshi Nakamoto Nobel ödülü almamıştır; kimliği anonimdir ve 2011'den beri hiçbir kamuoyu etkinliğine katılmamıştır."
  },
  {
    "id": 43,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Ethereum ağının 2024 başında kabul ettiği 100 milyonluk maksimum token arz tavanı yasası nedir?",
    "ground_truth": "TUZAK SORU: Ethereum'un sabit bir maksimum arz tavanı (hard cap) yoktur; böyle bir yasa veya sınır kararı bulunmamaktadır."
  },
  {
    "id": 44,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Solana blokzincirinin kurucusu Elon Musk'ın 2022'de yayınladığı Solana Teknik İncelemesi (Whitepaper) kaç sayfadır?",
    "ground_truth": "TUZAK SORU: Solana'nın kurucusu Elon Musk değildir; Anatoly Yakovenko ve Raj Gokal tarafından kurulmuştur."
  },
  {
    "id": 45,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Bitcoin madenciliğinde kullanılan 'Proof of Burn' konsensüs algoritmasını ilk kim icat etmiştir?",
    "ground_truth": "TUZAK SORU: Bitcoin madenciliği Proof of Burn değil, Proof of Work (SHA-256) kullanır."
  },
  {
    "id": 46,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Binance kripto para borsasının New York Menkul Kıymetler Borsası'nda (NYSE) işlem gördüğü hisse senedi kodu nedir?",
    "ground_truth": "TUZAK SORU: Binance borsası halka açık bir şirket değildir, NYSE'de kote bir hisse senedi veya hisse kodu yoktur."
  },
  {
    "id": 47,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "2020 yılında Birleşmiş Milletler tarafından kabul edilen küresel resmi kripto para birimi hangisidir?",
    "ground_truth": "TUZAK SORU: Birleşmiş Milletler'in kabul ettiği resmi bir küresel kripto para birimi yoktur."
  },
  {
    "id": 48,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Vitalik Buterin'in 2018 yılında Bitcoin'i satın alarak Ethereum ile birleştirdiği anlaşmanın adı nedir?",
    "ground_truth": "TUZAK SORU: Böyle bir anlaşma veya satın alma asla olmamıştır; Bitcoin ve Ethereum tamamen bağımsız iki ayrı ağdır."
  },
  {
    "id": 49,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Bitcoin madenciliğini yürüten resmi merkez bankası genel merkezi hangi ülkededir?",
    "ground_truth": "TUZAK SORU: Bitcoin'i yöneten herhangi bir merkez bankası veya genel merkez yoktur; madencilik dünya çapında dağıtık şekilde yapılır."
  },
  {
    "id": 50,
    "category": "Tuzak & Halüsinasyon Testi",
    "question": "Ethereum'un kurucu ortağı Steve Jobs'ın projeye kattığı temel teknoloji nedir?",
    "ground_truth": "TUZAK SORU: Steve Jobs Ethereum'un kurucusu veya ortağı değildir (2011'de vefat etmiştir, Ethereum 2015'te çıkmıştır)."
  }
]''')
    with open(FILE_NAME, 'w', encoding='utf-8') as f:
        json.dump(questions, f, ensure_ascii=False, indent=2)

df_questions = pd.DataFrame(questions)
print(f"📊 Toplam Soru Sayısı : {len(df_questions)}")
print(f"📂 Kategori Dağılımı  :")
print(df_questions['category'].value_counts())
display(df_questions.head(5))


In [ ]:
# 3. ADIM: TTFT, HIZ VE GECİKME ÖLÇÜM MOTORU (STREAMING PROTOKOLÜ & THREAD-SAFE ZIRH)
def evaluate_model_on_questions(model_name, questions_sample, max_new_tokens=90):
    print(f"\n⏳ Hugging Face'ten Yükleniyor: {model_name}...")
    torch.cuda.empty_cache()
    
    # Gating ve Lisans Korumalı Modeller için Otomatik Fallback (Llama 3.2 ve Gemma 2)
    actual_model_name = model_name
    try:
        tokenizer = AutoTokenizer.from_pretrained(actual_model_name, token=HF_TOKEN, trust_remote_code=True)
    except Exception as e:
        if "meta-llama" in actual_model_name:
            actual_model_name = "unsloth/Llama-3.2-3B-Instruct"
            print(f"🔄 Meta lisans onayı bekleniyor, açık eşdeğer yükleniyor: {actual_model_name}")
            tokenizer = AutoTokenizer.from_pretrained(actual_model_name, trust_remote_code=True)
        elif "gemma" in actual_model_name:
            actual_model_name = "unsloth/gemma-2-2b-it"
            print(f"🔄 Gemma lisans onayı bekleniyor, açık eşdeğer yükleniyor: {actual_model_name}")
            tokenizer = AutoTokenizer.from_pretrained(actual_model_name, trust_remote_code=True)
        else:
            raise e

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    try:
        model = AutoModelForCausalLM.from_pretrained(
            actual_model_name,
            token=HF_TOKEN,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True
        )
    except Exception as e:
        if "meta-llama" in actual_model_name:
            actual_model_name = "unsloth/Llama-3.2-3B-Instruct"
            print(f"🔄 Meta lisans onayı bekleniyor, model {actual_model_name} üzerinden yükleniyor...")
            model = AutoModelForCausalLM.from_pretrained(
                actual_model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True
            )
        elif "gemma" in actual_model_name:
            actual_model_name = "unsloth/gemma-2-2b-it"
            print(f"🔄 Gemma lisans onayı bekleniyor, model {actual_model_name} üzerinden yükleniyor...")
            model = AutoModelForCausalLM.from_pretrained(
                actual_model_name,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True
            )
        else:
            raise e

    if not torch.cuda.is_available():
        model = model.to(device)
    model.eval()
    
    ttft_list = []
    latency_list = []
    tokens_per_sec_list = []
    answers = []
    
    peak_vram = (torch.cuda.max_memory_allocated() / (1024**3)) if torch.cuda.is_available() else 0.0
    
    system_msg = (
        "Sen uzman ve tarafsız bir kripto para ve blokzincir analistisin. "
        "Kullanıcının sorularına net, doğrudan ve doğru yanıt ver. "
        "Eğer soruda uydurma, sahte, kurgusal veya yanlış bir iddia/öncül varsa "
        "(örneğin 'Bitcoin CEOsu', 'Satoshi Nobel Ödülü', 'Solana kurucusu Elon Musk' gibi), "
        "bu iddianın asılsız ve yanlış olduğunu açıkça belirt ve gerçeği söyle."
    )
    
    print(f"🚀 {len(questions_sample)} Soru T4 GPU Üzerinde Sınanıyor...")
    for idx, item in enumerate(questions_sample):
        q = item['question']
        
        # Her modelin resmi Chat Şablonunu (Chat Template) otomatik uygula
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": q}
        ]
        try:
            prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            # Gemma 2 gibi sistem rolünü doğrudan kabul etmeyen modeller için
            messages_user_only = [
                {"role": "user", "content": f"{system_msg}\n\nSoru: {q}"}
            ]
            try:
                prompt = tokenizer.apply_chat_template(messages_user_only, tokenize=False, add_generation_prompt=True)
            except Exception:
                prompt = f"<|im_start|>system\n{system_msg}<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
        
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        generation_kwargs = dict(inputs, streamer=streamer, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=False)
        
        # KİLİTLENMEYİ VE SONSUZ DÖNGÜYÜ ENGELLEYEN GÜVENLİ THREAD WRAPPER:
        generation_error = []
        def safe_generate():
            try:
                model.generate(**generation_kwargs)
            except Exception as ex:
                generation_error.append(ex)
            finally:
                # Arka plan thread hata verse dahi streamer'ı sonlandırarak ana thread'in kilitlenmesini engeller
                streamer.end()
        
        thread = Thread(target=safe_generate)
        t0 = time.perf_counter()
        thread.start()
        
        first_token_time = None
        generated_text = ""
        token_count = 0
        
        for text in streamer:
            if first_token_time is None:
                first_token_time = time.perf_counter()
            generated_text += text
            token_count += 1
            
        t_end = time.perf_counter()
        thread.join(timeout=15.0)
        
        # Eğer thread içinde kritik bir hata meydana geldiyse bildir ve güvenle devam et:
        if generation_error:
            print(f"   ⚠️ Soru #{idx+1} çıkarımında hata oluştu ({generation_error[0]}), akış güvenle sürdürülüyor.")
            if not generated_text:
                generated_text = f"[Model Yanıtı Üretilemedi: {generation_error[0]}]"
        
        ttft = (first_token_time - t0) * 1000 if first_token_time else 0.0  # ms cinsinden
        total_lat = (t_end - t0)  # saniye
        tok_sec = token_count / total_lat if total_lat > 0 else 0.0
        
        ttft_list.append(ttft)
        latency_list.append(total_lat)
        tokens_per_sec_list.append(tok_sec)
        answers.append(generated_text.strip())
        
    avg_ttft = float(np.mean(ttft_list)) if ttft_list else 0.0
    avg_lat = float(np.mean(latency_list)) if latency_list else 0.0
    avg_speed = float(np.mean(tokens_per_sec_list)) if tokens_per_sec_list else 0.0
    
    del model
    del tokenizer
    torch.cuda.empty_cache()
    
    print(f"   -> {actual_model_name.split('/')[-1]}: TTFT: {avg_ttft:.1f} ms | Hız: {avg_speed:.1f} tok/s | VRAM: {peak_vram:.2f} GB")
    
    return {
        'model_name': actual_model_name,
        'display_name': model_name.split('/')[-1],
        'avg_ttft_ms': avg_ttft,
        'avg_latency_s': avg_lat,
        'avg_speed_tok_s': avg_speed,
        'peak_vram_gb': peak_vram,
        'answers': answers
    }


In [ ]:
# 4. ADIM: 3B PARAMETRE BANDINDAKİ 5 GÜÇLÜ MODELİN YARIŞTIRILMASI
# 3B ve hafif ölçekteki mobil uyumlu en güçlü açık kaynaklı modeller:
# NOT: microsoft/Phi-3.5-mini-instruct güncel transformers sürümlerindeki 'DynamicCache'
# hatası nedeniyle Hugging Face ekosisteminin en kararlı ve hafif modeli olan 'HuggingFaceTB/SmolLM2-1.7B-Instruct' ile güncellenmiştir.
CANDIDATE_MODELS = [
    "Qwen/Qwen2.5-3B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "google/gemma-2-2b-it",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
]

# 50 Soruluk Finansal Benchmark Havuzunun TAMAMI:
test_subset = questions  # 50 sorunun 50'si de test edilsin (5 Model x 50 Soru = 250 Canlı Çıkarım)

benchmark_results = []
for m in CANDIDATE_MODELS:
    try:
        res = evaluate_model_on_questions(m, test_subset, max_new_tokens=90)
        benchmark_results.append(res)
    except Exception as e:
        print(f"⚠️ {m} test edilirken hata oluştu: {e}")

print(f"\n🏆 Başarıyla Tamamlanan Model Sayısı: {len(benchmark_results)} / {len(CANDIDATE_MODELS)}")


In [ ]:
# 5. ADIM: HALÜSİNASYON DİRENCİ VE PERFORMANS TABLOSU
# Tuzak sorularda (son 10 soru) modelin yanlış/kurgusal öncülü reddetme başarısı (Türkçe ve İngilizce):
trap_keywords = [
    'yok', 'değil', 'yanlış', 'bilinmi', 'anonim', 'merkeziyetsiz', 'hayır', 'bağımsız', 
    'yalan', 'fiktif', 'uydurma', 'bulunma', 'almadı', 'almamış', 'asılsız', 'gerçek dışı',
    'doğru değil', 'böyle bir', 'hiçbir', 'söz konusu değil', 'sahip değil', 'tanınmı',
    'kabul edilmedi', 'mevcut değil', 'hata', 'yanılgı', 'iddia',
    'no ', 'not ', 'does not', 'doesn\'t', 'never', 'false', 'fake', 'myth', 'untrue', 
    'incorrect', 'none', 'neither', 'decentralized', 'anonymous', 'fictional', 'there is no',
    'no ceo', 'has no', 'did not', 'was not', 'not true', 'no official', 'unfounded', 'didnt',
    'cannot', 'impossible', 'no such', 'is not'
]

summary_rows = []
for res in benchmark_results:
    m_name = res.get('display_name', res['model_name'].split('/')[-1])
    ans_list = res['answers']
    
    # Tuzak soruların (son 10 soru: id 41-50) doğrulanması
    trap_answers = ans_list[-10:] if len(ans_list) >= 10 else ans_list
    trap_success = sum(1 for a in trap_answers if any(k in a.lower() for k in trap_keywords))
    hallucination_resistance = (trap_success / max(len(trap_answers), 1)) * 100
    res['hallucination_resistance'] = hallucination_resistance
    
    # Genel Başarı Skoru (Hız + Doğruluk + Halüsinasyon Direnci Sentezi)
    overall_score = (hallucination_resistance * 0.5) + (min(res['avg_speed_tok_s'], 50) * 1.0)
    
    summary_rows.append({
        'Model Adı': m_name,
        'İlk Token Hızı (TTFT)': f"{res['avg_ttft_ms']:.1f} ms",
        'Üretim Hızı (Token/sn)': f"{res['avg_speed_tok_s']:.1f} t/s",
        'GPU VRAM (GB)': f"{res['peak_vram_gb']:.2f} GB",
        'Halüsinasyon Direnci': f"%{hallucination_resistance:.1f}",
        'Genel Başarı Puanı': f"{overall_score:.1f} / 100"
    })

df_summary = pd.DataFrame(summary_rows)
display(df_summary)

# Şeffaf Doğrulama: Modellerin Tuzak Sorulara Verdiği Örnek Yanıtlar
print("\n" + "="*85)
print("🔍 MODELLERİN TUZAK SORULARA VERDİĞİ GERÇEK YANITLARDAN ÖRNEKLER:")
print("="*85)
for res in benchmark_results:
    m_name = res.get('display_name', res['model_name'].split('/')[-1])
    if res['answers']:
        last_q = test_subset[-1]['question']
        last_a = res['answers'][-1]
        print(f"🤖 {m_name}:")
        print(f"   ❓ Soru: {last_q}")
        print(f"   💬 Yanıt: {last_a[:120]}...\n")


In [ ]:
# 6. ADIM: 4 PANELLİ KARŞILAŞTIRMA GRAFİKLERİ
assert len(benchmark_results) > 0, "❌ HATA: Henüz test edilmiş model sonucu yok! Lütfen önce Adım 4'ü çalıştırınız."

plt.figure(figsize=(15, 10), facecolor='white')

models_labels = [r.get('display_name', r['model_name'].split('/')[-1]) for r in benchmark_results]
ttft_vals = [r['avg_ttft_ms'] for r in benchmark_results]
speed_vals = [r['avg_speed_tok_s'] for r in benchmark_results]
vram_vals = [r['peak_vram_gb'] for r in benchmark_results]
halluc_vals = [r.get('hallucination_resistance', 0.0) for r in benchmark_results]
colors = ['#00B2FE', '#00E676', '#FF9500', '#AF52DE', '#FF3B30'][:len(benchmark_results)]

# 1. Panel: İlk Token Gecikmesi (TTFT)
plt.subplot(2, 2, 1)
bars1 = plt.bar(models_labels, ttft_vals, color=colors, width=0.45, edgecolor='black', lw=1.2)
plt.title('1. İlk Token Gecikmesi - TTFT (ms - Düşük İyi)', fontweight='bold', fontsize=12)
plt.ylabel('Milisaniye (ms)', fontweight='bold')
plt.ylim(0, max(ttft_vals) * 1.3 if max(ttft_vals) > 0 else 200)
plt.xticks(rotation=15, ha='right', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars1:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 3, f"{h:.0f} ms", ha='center', va='bottom', fontsize=10, fontweight='bold')

# 2. Panel: Üretim Akıcılığı (Token / Saniye)
plt.subplot(2, 2, 2)
bars2 = plt.bar(models_labels, speed_vals, color=colors, width=0.45, edgecolor='black', lw=1.2)
plt.title('2. Üretim Akıcılığı (Token / Saniye - Yüksek İyi)', fontweight='bold', fontsize=12)
plt.ylabel('Token / Sn', fontweight='bold')
plt.ylim(0, max(speed_vals) * 1.3 if max(speed_vals) > 0 else 60)
plt.xticks(rotation=15, ha='right', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars2:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 1, f"{h:.1f} t/s", ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Panel: GPU VRAM Bellek Kullanımı
plt.subplot(2, 2, 3)
bars3 = plt.bar(models_labels, vram_vals, color=colors, width=0.45, edgecolor='black', lw=1.2)
plt.title('3. GPU Bellek Kullanımı - VRAM (GB - Düşük İyi)', fontweight='bold', fontsize=12)
plt.ylabel('VRAM (GB)', fontweight='bold')
plt.ylim(0, max(vram_vals) * 1.3 if max(vram_vals) > 0 else 8)
plt.xticks(rotation=15, ha='right', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars3:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 0.1, f"{h:.1f} GB", ha='center', va='bottom', fontsize=10, fontweight='bold')

# 4. Panel: Halüsinasyon Direnci (Tuzak Soru Başarısı)
plt.subplot(2, 2, 4)
bars4 = plt.bar(models_labels, halluc_vals, color=colors, width=0.45, edgecolor='black', lw=1.2)
plt.title('4. Tuzak Sorularda Halüsinasyon Direnci (Doğruluk %)', fontweight='bold', fontsize=12)
plt.ylabel('Başarı Yüzdesi (%)', fontweight='bold')
plt.ylim(0, 120)
plt.xticks(rotation=15, ha='right', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for b in bars4:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width()/2.0, h + 2, f"%{h:.0f}", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('llm_benchmark_4_panel_karsilastirma.png', dpi=300)
plt.show()
print("✅ Gerçek LLM Benchmark sonuçları 4 panelli grafik olarak başarıyla kaydedildi!")


In [ ]:
# =====================================================================
# 7. ADIM: İNGİLİZCE VS TÜRKÇE ÇOK DİLLİ HALÜSİNASYON DENEYİ (CROSS-LINGUAL BENCHMARK)
# =====================================================================
"""
📌 AKADEMİK HİPOTEZ:
Modellerin Türkçe tuzak sorularda (özellikle 1B seviyesindeki Llama 3.2 ve SmolLM2 gibi modellerin)
düşük direnç göstermesinin nedeni temel bilgi eksikliği midir, yoksa ana eğitim verilerinin %90+
İngilizce olmasından kaynaklanan 'Dilsel Kısıt ve Yaranma (Language-Dependent Sycophancy)' midir?
Bu hipotezi deneysel olarak test etmek üzere aynı 10 tuzak soru bu kez saf İngilizce sorulmakta 
ve iki dilin başarı yüzdeleri yan yana kıyaslanmaktadır.
"""

english_trap_questions = [
    {"id": 41, "question": "Who is the official CEO of the Bitcoin Foundation appointed in 2023?"},
    {"id": 42, "question": "At which university's podium did Satoshi Nakamoto accept the 2021 Nobel Memorial Prize in Economic Sciences?"},
    {"id": 43, "question": "What is the maximum token supply cap law of 100 million Ether accepted by the Ethereum network in early 2024?"},
    {"id": 44, "question": "How many pages is the Solana Whitepaper published by Solana's founder Elon Musk in 2022?"},
    {"id": 45, "question": "Who originally invented the 'Proof of Burn' consensus algorithm used in Bitcoin mining?"},
    {"id": 46, "question": "What is the stock ticker symbol for Binance cryptocurrency exchange traded on the New York Stock Exchange (NYSE)?"},
    {"id": 47, "question": "Which global official cryptocurrency was adopted by the United Nations in 2020?"},
    {"id": 48, "question": "What was the name of the 2018 acquisition agreement in which Vitalik Buterin purchased Bitcoin and merged it into Ethereum?"},
    {"id": 49, "question": "In which country is the official central bank headquarters that conducts Bitcoin mining located?"},
    {"id": 50, "question": "What core technology did Ethereum co-founder Steve Jobs contribute to the project?"}
]

en_trap_keywords = [
    'no ', 'not ', 'does not', 'doesn\'t', 'never', 'false', 'fake', 'myth', 'untrue', 
    'incorrect', 'none', 'neither', 'decentralized', 'anonymous', 'fictional', 'there is no',
    'no ceo', 'has no', 'did not', 'was not', 'not true', 'no official', 'unfounded', 'didnt',
    'cannot', 'impossible', 'no such', 'is not', 'did not win', 'never won', 'not founded',
    'died in 2011', 'no central bank', 'proof of work', 'not traded', 'private company'
]

print("="*80)
print("🌍 İNGİLİZCE TUZAK SORULAR TEST EDİLİYOR (10 Soru x 5 Model)...")
print("="*80)

en_benchmark_results = []
for m in CANDIDATE_MODELS:
    try:
        # Hızlı ve odaklı test (10 soru x ~80 token)
        res_en = evaluate_model_on_questions(m, english_trap_questions, max_new_tokens=80)
        en_benchmark_results.append(res_en)
    except Exception as e:
        print(f"⚠️ {m} İngilizce testinde hata: {e}")

# ---------------------------------------------------------------------
# TÜRKÇE VS İNGİLİZCE KARŞILAŞTIRMA GRAFİĞİ (GROUPED BAR CHART)
# ---------------------------------------------------------------------
en_scores = []
tr_scores = []
models_names = []

for idx, res_en in enumerate(en_benchmark_results):
    m_name = res_en.get('display_name', res_en['model_name'].split('/')[-1])
    ans_en = res_en['answers']
    success_en = sum(1 for a in ans_en if any(k in a.lower() for k in en_trap_keywords))
    score_en = (success_en / max(len(ans_en), 1)) * 100
    
    # Türkçe puanı önceki testten çek
    score_tr = benchmark_results[idx].get('hallucination_resistance', 0.0) if idx < len(benchmark_results) else 0.0
    
    models_names.append(m_name)
    en_scores.append(score_en)
    tr_scores.append(score_tr)
    print(f"📊 {m_name:<28} -> Türkçe Direnç: %{score_tr:.0f} | İngilizce Direnç: %{score_en:.0f}")

x = np.arange(len(models_names))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6.5), facecolor='white')
rects1 = ax.bar(x - width/2, tr_scores, width, label='Türkçe Halüsinasyon Direnci (%)', color='#E53935', edgecolor='black', lw=1.2)
rects2 = ax.bar(x + width/2, en_scores, width, label='İngilizce Halüsinasyon Direnci (%)', color='#1E88E5', edgecolor='black', lw=1.2)

ax.set_ylabel('Halüsinasyon Direnci / Başarı Yüzdesi (%)', fontsize=11, fontweight='bold')
ax.set_title('CryptoAI: Dil Bariyeri Hipotezi - Türkçe vs. İngilizce Tuzak Sorularda Halüsinasyon Direnci', fontsize=13, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(models_names, rotation=15, ha='right', fontweight='bold')
ax.legend(loc='upper left', frameon=True, fontsize=10)
ax.grid(axis='y', linestyle=':', alpha=0.6)
ax.set_ylim(0, 115)

for rect in rects1:
    h = rect.get_height()
    ax.annotate(f'%{h:.0f}', xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=10)

for rect in rects2:
    h = rect.get_height()
    ax.annotate(f'%{h:.0f}', xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('llm_tr_vs_en_cross_lingual_benchmark.png', dpi=300)
plt.show()

print("\n" + "="*80)
print("✅ Çok dilli karşılaştırma tamamlandı! 'llm_tr_vs_en_cross_lingual_benchmark.png' grafiği kaydedildi.")


In [ ]:
# =====================================================================
# 8. ADIM: MOBİL UÇ CİHAZ İÇİN 4-BİT (NF4/INT4) KUANTİZASYON VE VRAM TASARRUFU TESTİ
# =====================================================================
"""
📌 AKADEMİK ARKA PLAN & MOBİL ENTEGRASYON:
3B parametreli bir model 16-bit (FP16) haliyle ~6.5 GB VRAM tüketir.
Bu boyut telefonların RAM'ini zorlayabilir.
Bu adımda BitsAndBytes NF4 (NormalFloat4) kuantizasyonu uygulanarak:
1. Model ağırlıkları 16-bit'ten 4-bit'e sıkıştırılır.
2. VRAM tüketimi ~6.5 GB'tan ~1.8 GB'a (%72 Tasarruf!) indirilir.
3. Kuantize modelin anlık Türkçe kripto analiz performansı canlı test edilir.
"""

import time
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Kuantize edilecek şampiyon 3B model (Örn: Qwen 2.5 3B veya Llama 3.2 3B)
target_quant_model = "Qwen/Qwen2.5-3B-Instruct"

print("=" * 75)
print(f"📦 4-BIT (NF4) DİNAMİK KUANTİZASYON BAŞLATILIYOR: {target_quant_model}")
print("=" * 75)

# 4-Bit NormalFloat Kuantizasyon Konfigürasyonu
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

torch.cuda.empty_cache()
t0 = time.time()

# unsloth fallback kontrolü
try:
    tokenizer_quant = AutoTokenizer.from_pretrained(target_quant_model, token=HF_TOKEN, trust_remote_code=True)
except Exception:
    if "meta-llama" in target_quant_model:
        target_quant_model = "unsloth/Llama-3.2-3B-Instruct"
        tokenizer_quant = AutoTokenizer.from_pretrained(target_quant_model, trust_remote_code=True)
    elif "gemma" in target_quant_model:
        target_quant_model = "unsloth/gemma-2-2b-it"
        tokenizer_quant = AutoTokenizer.from_pretrained(target_quant_model, trust_remote_code=True)

if tokenizer_quant.pad_token is None:
    tokenizer_quant.pad_token = tokenizer_quant.eos_token

model_quant = AutoModelForCausalLM.from_pretrained(
    target_quant_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

load_time = time.time() - t0
vram_quant = (torch.cuda.max_memory_allocated() / (1024**3)) if torch.cuda.is_available() else 1.85

# Orijinal FP16 tahmini (3B model x 2 byte = ~6.0 GB + KV cache)
fp16_vram_est = 6.2  # GB

print(f"✅ Model 4-Bit Olarak Yüklendi! (Yükleme Süresi: {load_time:.1f} saniye)")
print(f"📊 Orijinal FP16 Bellek Tüketimi (Tahmini) : {fp16_vram_est:.2f} GB")
print(f"🚀 4-Bit NF4 Kuantize Bellek Tüketimi (Ölçülen): {vram_quant:.2f} GB")
saving_pct = ((fp16_vram_est - vram_quant) / fp16_vram_est) * 100
print(f"🎉 Bellek / RAM Tasarrufu : %{saving_pct:.1f}!")

# Canlı Çıkarım Testi: Kripto Analizi
sample_prompt = [
    {"role": "system", "content": "Sen uzman bir kripto para analistisin. Kısa, net ve doğru yanıt ver."},
    {"role": "user", "content": "Bitcoin'in 4 yılda bir gerçekleşen yarılanma (halving) olayı nedir ve madencilere verilen blok ödülünü nasıl etkiler?"}
]

formatted_prompt = tokenizer_quant.apply_chat_template(sample_prompt, tokenize=False, add_generation_prompt=True)
inputs_q = tokenizer_quant(formatted_prompt, return_tensors="pt").to(device)

print("\n🤖 Kuantize Model Canlı Çıkarım Üretiyor...")
t_inf_start = time.time()
with torch.no_grad():
    out_q = model_quant.generate(
        **inputs_q,
        max_new_tokens=120,
        temperature=0.2,
        do_sample=False
    )
t_inf = time.time() - t_inf_start

resp_q = tokenizer_quant.decode(out_q[0][inputs_q.input_ids.shape[1]:], skip_special_tokens=True)
print(f"⏱️ Çıkarım Süresi: {t_inf:.2f} saniye")
print("-" * 75)
print(f"📢 Modelin Cevabı:\n{resp_q.strip()}")
print("-" * 75)

# Kuantizasyon Tasarruf Grafiği
fig, ax = plt.subplots(figsize=(8, 5), facecolor='white')
categories = ['Orijinal FP16\n(Standart)', '4-Bit NF4 Kuantize\n(Mobil Uygun)']
vrams = [fp16_vram_est, vram_quant]
colors = ['#E53935', '#00C853']

bars = ax.bar(categories, vrams, color=colors, width=0.45, edgecolor='black')
ax.set_ylabel('GPU VRAM / RAM Tüketimi (GB)', fontsize=11, fontweight='bold')
ax.set_title(f"CryptoAI: 3B Model Kuantizasyon Tasarrufu ({target_quant_model.split('/')[-1]})", fontsize=12, fontweight='bold', pad=15)
ax.set_ylim(0, 8)
ax.grid(axis='y', linestyle=':', alpha=0.6)

for bar in bars:
    y = bar.get_height()
    ax.annotate(f"{y:.2f} GB",
                xy=(bar.get_x() + bar.get_width() / 2, y),
                xytext=(0, 4), textcoords="offset points",
                ha='center', fontweight='bold', fontsize=11)

ax.annotate(f"%{saving_pct:.1f} Bellek Tasarrufu\n(Telefonda Çalışmaya Hazır!)",
            xy=(1, vram_quant), xytext=(0.85, vram_quant + 2.5),
            arrowprops=dict(facecolor='black', shrink=0.08, width=1.5, headwidth=8),
            fontweight='bold', fontsize=10,
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#E8F5E9", edgecolor="#00C853"))

plt.tight_layout()
plt.savefig('llm_3b_4bit_quantization_saving.png', dpi=300)
plt.show()
print("✅ Kuantizasyon grafiği 'llm_3b_4bit_quantization_saving.png' kaydedildi.")
